# Notebook 03b · Transferability — Hanoi & HCMC (**Sentinel-2 manual stack**)

The manual-stack counterpart of `03_Transferability_Vietnam_S2_median.ipynb`.
Same points, same spatial-block design, same evaluation — **only the
predictor changes**: 10 S2 median bands become `10 * n_dates` stack bands (30
for Hanoi, 10 for HCMC), built and extracted by
`00c_S2_Extraction_Vietnam_Stack_Manual.ipynb`.

## No zero-shot

Milan's own stack model (`output/milan/clms/stack/`) has 40 bands (4 dates).
Hanoi's stack is 30 bands, HCMC's is 10 — neither matches Milan's band count,
so a Milan-trained stack classifier cannot be applied to either city's
composite. **Only Scenario B (local retrain) exists for the stack.** This is a
structural limit of the predictor, not a decision to skip the comparison; Cell
1 prints the three band counts side by side.

## Pipeline

| Cell | Does |
|---|---|
| 1 | Config; print Milan's stack dimensionality against each city's |
| 2 | Helpers — spatial blocks (same as 01c / 02 / 03) |
| 3 | Per city: spatial-block hyperparameter search, evaluate on held-out test |
| 4 | Full-raster export (GEE batch export to Drive, 10 m) |
| 5 | Difference statistics against the GHSL benchmark, full raster |
| 6 | Summary; comparison against the median run |

> Prerequisite: run `00c_S2_Extraction_Vietnam_Stack_Manual.ipynb` first.

## Cell 1 · Imports & configuration

In [1]:
# Interpreter + PROJ check -- same fix as notebooks 00b/02/03. Must run before
# rasterio/geopandas import.
import sys, os
for _v in ('PROJ_LIB', 'PROJ_DATA'):
    os.environ.pop(_v, None)
import rasterio.crs
import pyproj
rasterio.crs.CRS.from_epsg(32648)
pyproj.CRS.from_epsg(32648)
print('interpreter :', sys.executable)
print('PROJ check  : OK (EPSG:32648 resolves in rasterio and pyproj)')

interpreter : C:\ProgramData\anaconda3\python.exe
PROJ check  : OK (EPSG:32648 resolves in rasterio and pyproj)


In [2]:
# Earth Engine project id comes from .env (copy .env.example to .env).
# Authentication is a one-off per machine: run `earthengine authenticate`.
import ee
from gee_init import init_gee

init_gee()

import json
import time
import warnings

import geopandas as gpd
import numpy as np
import rasterio
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
from rasterio.windows import from_bounds
from scipy.spatial import cKDTree
from shapely.geometry import box
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

from s2_utils import build_composite

warnings.filterwarnings('ignore')

# -- Config, matching notebooks 01c/02/03 exactly -------------------------------
RANDOM_STATE  = 42
N_JOBS        = -1
N_CV_FOLDS    = 5
N_ITER_RF     = 30
BLOCK_SIZES   = [500, 1000, 2000]
BUFFER_M      = 250
S2_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'
BBOX_HALF     = 15000

PARAM_SPACE = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
    'bootstrap':         [True, False],
    'max_samples':       [None, 0.7, 0.8, 0.9],
}

# -- City centres and manual stack dates -- identical to 00c ---------------------
CITIES = {
    'Hanoi': {
        'centre_lonlat': (105.85, 21.03),
        'utm_crs':       'EPSG:32648',
        'dates':         ['2018-01-19', '2018-10-31', '2018-12-20'],
        'ghsl_path':     './data/GHSL_2018_Hanoi_UTM48N.tif',
    },
    'HCMC': {
        'centre_lonlat': (106.70, 10.78),
        'utm_crs':       'EPSG:32648',
        'dates':         ['2018-02-12'],
        'ghsl_path':     './data/GHSL_2018_HCMC_UTM48N.tif',
    },
}

def input_dir(city):
    return f'./output/transfer_vietnam/{city.lower()}/stack_manual'

def comparison_dir():
    d = './output/transfer/stack_manual'
    os.makedirs(d, exist_ok=True)
    return d

DRIVE_FOLDER = 'GEE_IMD_exports'

# Milan's own stack dimensionality -- printed so the "no zero-shot" header
# claim is checked against the actual Milan run, not just asserted.
with open('./output/milan/clms/stack/model_metadata_S2.json') as f:
    milan_stack_meta = json.load(f)
print(f"Milan stack model: {len(milan_stack_meta['band_names'])} bands "
      f"({len(milan_stack_meta['selected_dates'])} dates)")
for city, cfg in CITIES.items():
    print(f"  {city:6s} stack: {10 * len(cfg['dates'])} bands "
          f"({len(cfg['dates'])} date{'s' if len(cfg['dates']) != 1 else ''}) -- differs from Milan, no zero-shot.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


GEE initialised (project: imd-vietnam).


Milan stack model: 40 bands (4 dates)
  Hanoi  stack: 30 bands (3 dates) -- differs from Milan, no zero-shot.
  HCMC   stack: 10 bands (1 date) -- differs from Milan, no zero-shot.


---
## Cell 2 · Helper functions

Spatial block assignment, CV folds and the train/test buffer -- identical to `03_Transferability_Vietnam_S2_median.ipynb` Cell 2.

In [3]:
def assign_blocks(coords, block_size):
    bx = (coords[:, 0] // block_size).astype(int)
    by = (coords[:, 1] // block_size).astype(int)
    return np.array([f'{x}_{y}' for x, y in zip(bx, by)])

def blocks_to_folds(block_ids, n_folds, seed):
    rng    = np.random.RandomState(seed)
    unique = rng.permutation(np.unique(block_ids))
    bmap   = {b: i % n_folds for i, b in enumerate(unique)}
    return np.array([bmap[b] for b in block_ids])

def make_spatial_splits(point_folds, n_folds):
    all_idx = np.arange(len(point_folds))
    return [(all_idx[point_folds != f], all_idx[point_folds == f])
            for f in range(n_folds)]

def apply_buffer(train_coords, test_coords, buffer_m):
    if len(train_coords) == 0 or buffer_m == 0:
        return np.ones(len(test_coords), dtype=bool)
    dists, _ = cKDTree(train_coords).query(test_coords, k=1)
    return dists > buffer_m

print('Helper functions defined.')

Helper functions defined.


---
## Cell 3 · Per city: spatial-block hyperparameter search, evaluate on held-out test

Scenario B only (local retrain) -- see header.

In [4]:
city_results = {}

for city, cfg in CITIES.items():
    print(f"\n{'='*60}\n  {city}  --  stack dates: {cfg['dates']}\n{'='*60}")
    t0 = time.time()

    in_dir = input_dir(city)
    gdf_tr = gpd.read_file(f'{in_dir}/samples_{city}_train.gpkg')
    gdf_te = gpd.read_file(f'{in_dir}/samples_{city}_test.gpkg')
    with open(f'{in_dir}/s2_stack_manual_metadata.json') as f:
        in_meta = json.load(f)
    band_names = in_meta['band_names']
    print(f"  loaded {len(gdf_tr)} train + {len(gdf_te)} test points "
          f"({len(band_names)} bands, from 00c)")

    X_tr = gdf_tr[band_names].values.astype(np.float32)
    y_tr = gdf_tr['IMD'].values.astype(np.float32)
    X_te = gdf_te[band_names].values.astype(np.float32)
    y_te = gdf_te['IMD'].values.astype(np.float32)

    gdf_tr_utm = gdf_tr.to_crs(cfg['utm_crs'])
    coords_tr  = np.column_stack([gdf_tr_utm.geometry.x, gdf_tr_utm.geometry.y])

    # -- spatial N_CV_FOLDS-fold CV hyperparameter search across BLOCK_SIZES --
    best_rmse_tune, best_search, best_cv_block = np.inf, None, None
    for bs in BLOCK_SIZES:
        bids = assign_blocks(coords_tr, bs)
        pf = blocks_to_folds(bids, N_CV_FOLDS, RANDOM_STATE)
        cv_splits = make_spatial_splits(pf, N_CV_FOLDS)
        search = RandomizedSearchCV(
            estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
            param_distributions=PARAM_SPACE, n_iter=N_ITER_RF, cv=cv_splits,
            scoring='neg_root_mean_squared_error', n_jobs=N_JOBS,
            random_state=RANDOM_STATE, refit=True)
        search.fit(X_tr, y_tr)
        cv_rmse = -search.best_score_
        print(f"    {bs}m spatial CV -> RMSE={cv_rmse:.3f}")
        if cv_rmse < best_rmse_tune:
            best_rmse_tune, best_search, best_cv_block = cv_rmse, search, bs

    local_rf = best_search.best_estimator_
    print(f"  BEST: block={best_cv_block}m  CV_RMSE={best_rmse_tune:.3f}")
    print(f"  params={best_search.best_params_}")

    # -- evaluate on the held-out test set ------------------------------------
    y_pred = np.clip(local_rf.predict(X_te), 0, 100)
    metrics = {
        'RMSE': float(np.sqrt(mean_squared_error(y_te, y_pred))),
        'MAE':  float(mean_absolute_error(y_te, y_pred)),
        'R2':   float(r2_score(y_te, y_pred)),
        'Bias': float(np.mean(y_te - y_pred)),   # observed - predicted
    }
    print(f"  TEST: RMSE={metrics['RMSE']:.2f}  MAE={metrics['MAE']:.2f}  "
          f"R2={metrics['R2']:.3f}  Bias={metrics['Bias']:+.2f}")

    city_results[city] = {
        'band_names': band_names, 'model': local_rf,
        'n_train': len(gdf_tr), 'n_test': len(gdf_te),
        'best_cv_block_m': best_cv_block, 'cv_rmse_tune': best_rmse_tune,
        'best_params': best_search.best_params_, 'test_metrics': metrics,
        'elapsed_s': round(time.time() - t0, 1),
    }

print('\n=== TEST METRICS ===')
for city, r in city_results.items():
    m = r['test_metrics']
    print(f"{city:6s} ({len(r['band_names'])} bands, block={r['best_cv_block_m']}m): "
          f"RMSE={m['RMSE']:.2f} MAE={m['MAE']:.2f} R2={m['R2']:.3f} Bias={m['Bias']:+.2f}")


  Hanoi  --  stack dates: ['2018-01-19', '2018-10-31', '2018-12-20']
  loaded 2387 train + 895 test points (30 bands, from 00c)


    500m spatial CV -> RMSE=25.055


    1000m spatial CV -> RMSE=25.053


    2000m spatial CV -> RMSE=24.874
  BEST: block=2000m  CV_RMSE=24.874
  params={'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_samples': None, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': False}
  TEST: RMSE=23.35  MAE=18.32  R2=0.549  Bias=-1.78

  HCMC  --  stack dates: ['2018-02-12']
  loaded 2362 train + 887 test points (10 bands, from 00c)


    500m spatial CV -> RMSE=22.950


    1000m spatial CV -> RMSE=23.397


    2000m spatial CV -> RMSE=23.074
  BEST: block=500m  CV_RMSE=22.950
  params={'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_samples': 0.8, 'max_features': 0.5, 'max_depth': 10, 'bootstrap': True}
  TEST: RMSE=22.13  MAE=17.27  R2=0.621  Bias=-1.70

=== TEST METRICS ===
Hanoi  (30 bands, block=2000m): RMSE=23.35 MAE=18.32 R2=0.549 Bias=-1.78
HCMC   (10 bands, block=500m): RMSE=22.13 MAE=17.27 R2=0.621 Bias=-1.70


---
## Cell 4 · Full-raster export (GEE batch export, 10 m)

Classifies every pixel in each 30 km AOI. Uses `ee.batch.Export.image.toDrive` -- the same path notebook 02 Cell 7 uses for every other prediction raster in this project -- because the synchronous download path hits this account's compute-quota memory ceiling at 10 m over a many-band image. The final model is trained on **all** extracted points (train+test combined), matching that same convention. Skips the export if the raster already exists locally.

In [5]:
def gee_rf_params(band_names, params):
    # sklearn RandomForestRegressor params -> ee.Classifier.smileRandomForest kwargs.
    mf = params.get('max_features', 'sqrt')
    if mf == 'sqrt':
        vps = max(1, int(np.sqrt(len(band_names))))
    elif mf == 'log2':
        vps = max(1, int(np.log2(len(band_names))))
    else:
        vps = max(1, int(float(mf) * len(band_names)))
    max_samples = params.get('max_samples')
    bag = float(max_samples) if max_samples is not None else 0.8
    return dict(numberOfTrees=int(params['n_estimators']), variablesPerSplit=vps,
                minLeafPopulation=int(params['min_samples_leaf']), bagFraction=bag,
                seed=RANDOM_STATE)

In [6]:
def export_and_download(city, cfg, r):
    out_tif = f'{input_dir(city)}/IMD_{city}_10m_localrf_S2stack_manual.tif'
    if os.path.exists(out_tif):
        print(f'{city}: raster already present at {out_tif} -- skipping export.')
        return out_tif

    band_names = r['band_names']
    gdf = gpd.read_file(f'{input_dir(city)}/samples_{city}_stack_all.gpkg')

    lon, lat = cfg['centre_lonlat']
    centre_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy([lon], [lat]),
                                   crs='EPSG:4326').to_crs(cfg['utm_crs'])
    cx, cy = centre_gdf.geometry.x[0], centre_gdf.geometry.y[0]
    bbox_utm  = box(cx - BBOX_HALF, cy - BBOX_HALF, cx + BBOX_HALF, cy + BBOX_HALF)
    bbox_4326 = gpd.GeoSeries([bbox_utm], crs=cfg['utm_crs']).to_crs('EPSG:4326').iloc[0]
    aoi_ee    = ee.Geometry.Polygon([[list(c) for c in bbox_4326.exterior.coords]])

    img, proj, img_bands = build_composite(S2_COLLECTION, cfg['dates'], aoi_ee, method='stack')
    # GEE feature properties are named, not positional -- only the SET of bands
    # needs to match between the trained model and the freshly-built image.
    assert set(img_bands) == set(band_names), (
        f'{city}: composite bands disagree with the trained model.')

    feats = [ee.Feature(ee.Geometry.Point([row.geometry.x, row.geometry.y]),
                         {**{b: float(row[b]) for b in band_names}, 'IMD': float(row['IMD'])})
             for _, row in gdf.iterrows()]
    fc = ee.FeatureCollection(feats)

    clf = (ee.Classifier.smileRandomForest(**gee_rf_params(band_names, r['best_params']))
           .setOutputMode('REGRESSION')
           .train(features=fc, classProperty='IMD', inputProperties=band_names))
    pred_img = img.classify(clf).rename('IMD_predicted').clamp(0, 100)

    fname = f'IMD_{city}_10m_localrf_S2stack_manual'
    task = ee.batch.Export.image.toDrive(
        image=pred_img, description=fname, folder=DRIVE_FOLDER,
        fileNamePrefix=fname, region=aoi_ee, scale=10, crs=cfg['utm_crs'],
        maxPixels=1e10, fileFormat='GeoTIFF')
    task.start()
    print(f'{city}: submitted batch export task (id={task.id})')
    state = 'READY'
    while state not in ('COMPLETED', 'FAILED', 'CANCELLED'):
        time.sleep(15)
        state = task.status()['state']
        print(f'  {city}: {state}')
    if state != 'COMPLETED':
        raise RuntimeError(f'{city}: export {state} -- {task.status()}')

    # Download via Earth Engine's own OAuth credentials -- already scoped for
    # Drive, since EE authentication includes the drive scope.
    creds = ee.data.get_persistent_credentials()
    drive  = gbuild('drive', 'v3', credentials=creds)
    resp   = drive.files().list(q=f"name = '{fname}.tif' and trashed = false",
                                 fields='files(id, createdTime)').execute()
    files  = sorted(resp.get('files', []), key=lambda f: f['createdTime'], reverse=True)
    if not files:
        raise RuntimeError(f'{city}: export completed but {fname}.tif not found on Drive.')
    request = drive.files().get_media(fileId=files[0]['id'])
    with open(out_tif, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print(f'{city}: downloaded -> {out_tif}')
    return out_tif


for city, cfg in CITIES.items():
    city_results[city]['raster_path'] = export_and_download(city, cfg, city_results[city])

Hanoi: raster already present at ./output/transfer_vietnam/hanoi/stack_manual/IMD_Hanoi_10m_localrf_S2stack_manual.tif -- skipping export.
HCMC: raster already present at ./output/transfer_vietnam/hcmc/stack_manual/IMD_HCMC_10m_localrf_S2stack_manual.tif -- skipping export.


---
## Cell 5 · Difference statistics against the GHSL benchmark

Reported as **benchmark minus predicted** -- the same convention as this notebook's own point-level Bias (Cell 3): positive means the raster under-predicts GHSL on average. The results-grid figure used in the deck plots **predicted minus benchmark** instead; both are printed below, labelled, rather than silently picking one.

In [7]:
def diff_vs_ghsl(pred_path, ghsl_path):
    with rasterio.open(pred_path) as src:
        pred = src.read(1).astype('float32')
        pred[(pred < 0) | (pred > 100)] = np.nan
        bounds, shape = src.bounds, src.shape

    with rasterio.open(ghsl_path) as src:
        win = from_bounds(*bounds, transform=src.transform)
        bench = src.read(1, window=win, out_shape=shape, boundless=True).astype('float32')
        nodata = src.nodata
        if nodata is not None:
            bench[bench == nodata] = np.nan
        bench[(bench < 0) | (bench > 100)] = np.nan

    valid = ~np.isnan(pred) & ~np.isnan(bench)
    diff = bench[valid] - pred[valid]   # benchmark - predicted, see header
    return {
        'n_valid_px':       int(valid.sum()),
        'mean_diff_pp':     round(float(diff.mean()), 2),
        'mean_abs_diff_pp': round(float(np.abs(diff).mean()), 2),
        'std_diff_pp':      round(float(diff.std()), 2),
    }


raster_diff = {}
for city, cfg in CITIES.items():
    r = city_results[city]
    d = diff_vs_ghsl(r['raster_path'], cfg['ghsl_path'])
    raster_diff[city] = d
    print(f"{city:6s} (n={d['n_valid_px']:,} valid px)")
    print(f"  Benchmark - Predicted : {d['mean_diff_pp']:+.2f} pp mean, "
          f"{d['mean_abs_diff_pp']:.2f} pp mean abs, {d['std_diff_pp']:.2f} pp std")
    print(f"  Predicted - Benchmark : {-d['mean_diff_pp']:+.2f} pp "
          "(sign used by the results-grid figure's difference panels)")

Hanoi  (n=8,418,962 valid px)
  Benchmark - Predicted : -10.68 pp mean, 14.74 pp mean abs, 17.54 pp std
  Predicted - Benchmark : +10.68 pp (sign used by the results-grid figure's difference panels)


HCMC   (n=8,197,886 valid px)
  Benchmark - Predicted : -9.09 pp mean, 16.11 pp mean abs, 19.96 pp std
  Predicted - Benchmark : +9.09 pp (sign used by the results-grid figure's difference panels)


---
## Cell 6 · Summary

Saved in the same shape as `03`'s `transfer_summary.json`, minus `zero_shot` (does not exist for this predictor -- see header).

In [8]:
summary = {}
for city in CITIES:
    r = city_results[city]
    summary[city] = {
        'local_rf':            r['test_metrics'],
        'best_cv_block_m':     r['best_cv_block_m'],
        'cv_rmse_tune':        r['cv_rmse_tune'],
        'n_samples_train':     r['n_train'],
        'n_samples_test':      r['n_test'],
        'n_bands':             len(r['band_names']),
        'dates':               CITIES[city]['dates'],
        'raster_diff_vs_ghsl': raster_diff[city],
    }

with open(f'{comparison_dir()}/stack_manual_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Saved -> {comparison_dir()}/stack_manual_summary.json')

# -- Compare against the median run, for convenience only ------------------
# Reads 03's own output; does not alter it. Only runs if that notebook has
# already been executed on this machine.
median_summary_path = './output/transfer/median/transfer_summary.json'
if os.path.exists(median_summary_path):
    with open(median_summary_path) as f:
        median_summary = json.load(f)
    print('\n=== Stack vs median, local retrain, held-out test points ===')
    for city in CITIES:
        s_rmse = summary[city]['local_rf']['RMSE']
        m_rmse = median_summary[city]['local_rf']['RMSE']
        better = 'better' if s_rmse < m_rmse else 'worse'
        print(f"{city:6s} stack RMSE={s_rmse:.2f}  median RMSE={m_rmse:.2f}  "
              f"(stack {better} by {abs(s_rmse - m_rmse):.2f} pp)")
else:
    print(f'\n(median summary not found at {median_summary_path} -- skipping comparison)')

Saved -> ./output/transfer/stack_manual/stack_manual_summary.json

(median summary not found at ./output/transfer/median/transfer_summary.json -- skipping comparison)
